In [0]:
# Databricks notebook source

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window


In [0]:
bronze_sales_df = spark.table("bronze_sales")

### The Silver Layer applies business and data quality rules.

For this dataset, common retail cleaning rules are:

1. Convert invoice date to timestamp.
2. Remove records with missing invoice number.
3. Remove records with missing product code.
4. Remove records with missing or invalid customer ID.
5. Remove negative or zero quantity.
6. Remove negative or zero unit price.
7. Standardize text fields.
9. Calculate transaction amount.
10. Remove duplicates.

Flag cancelled invoices separately.

In [0]:
#Clean and Standardize

# Standardize text fields
silver_sales_df = (
    bronze_sales_df
    .withColumn("InvoiceNo", F.trim(F.col("InvoiceNo")))
    .withColumn("StockCode", F.trim(F.col("StockCode")))
    .withColumn("Description", F.trim(F.col("Description")))
    .withColumn("Country", F.trim(F.col("Country")))
    .withColumn("PaymentMethod", F.trim(F.col("PaymentMethod")))
    .withColumn("Category", F.trim(F.col("Category")))
    .withColumn("SalesChannel", F.trim(F.col("SalesChannel")))
    .withColumn("ReturnStatus", F.trim(F.col("ReturnStatus")))
    .withColumn("ShipmentProvider", F.trim(F.col("ShipmentProvider")))
    .withColumn("WarehouseLocation", F.trim(F.col("WarehouseLocation")))
    .withColumn("OrderPriority", F.trim(F.col("OrderPriority")))

    # Convert CustomerID to long and correct numerical data types
    .withColumn("CustomerID", F.regexp_replace(F.col("CustomerID"), r"^(\d+)\.(\d+)$", "$1").cast("long"))
    .withColumn("Discount", F.col("Discount").cast(DoubleType()))
    .withColumn("UnitPrice", F.col("UnitPrice").cast(DoubleType()))
    .withColumn("Quantity", F.col("Quantity").cast(IntegerType()))
    .withColumn("ShippingCost", F.col("ShippingCost").cast(DoubleType()))

    # Convert invoice date
    .withColumn("InvoiceDate", F.coalesce(
        #nvl expects only two arguments, so we used coalesce
        F.try_to_date(F.col("InvoiceDate"), "MM/dd/yyyy HH:mm"),
        F.try_to_date(F.col("InvoiceDate"), "MM-dd-yyyy HH:mm"),
        F.try_to_date(F.col("InvoiceDate"), "yyyy-MM-dd HH:mm"),
        F.try_to_date(F.col("InvoiceDate"), "yyyy/MM/dd HH:mm"),
        F.try_to_date(F.col("InvoiceDate"), "yyyyMMdd HH:mm"),
        F.try_to_date(F.col("InvoiceDate"), "dd-MM-yyyy HH:mm"))
    )

    # numeric standerdisation
    .withColumn("UnitPrice", F.round(F.col("UnitPrice"), 2))
    .withColumn("Discount", F.round(F.col("Discount"), 2))
    .withColumn("ShippingCost", F.round(F.col("ShippingCost"), 2))
    .withColumn("Quantity", F.round(F.col("Quantity"), 0))
    .withColumn("InvoiceNo", F.regexp_replace(F.col("InvoiceNo"), r"^\D*(\d+)\D*$", "$1"))

    # cancelled invoice indicator
    .withColumn("InvoiceCancelledInd", F.when(F.col("InvoiceNo").startswith("C"), True).otherwise(False))
    
    # Calculate transactions
    .withColumn("Sales", F.round(F.col("Quantity") * F.col("UnitPrice"), 2))
    .withColumn("Profit", F.round(F.col("Sales") * (1 - F.col("Discount")), 2))

    # Date fields
    .withColumn("InvoiceDate", F.col("InvoiceDate").cast(DateType()))
    .withColumn("InvoiceMonth", F.month(F.col("InvoiceDate")))
    .withColumn("InvoiceWeek", F.weekofyear(F.col("InvoiceDate")))
    .withColumn("InvoiceDay", F.dayofmonth(F.col("InvoiceDate")))
    .withColumn("InvoiceDayofWeek", F.dayofweek(F.col("InvoiceDate")))
    .withColumn("InvoiceYear", F.year(F.col("InvoiceDate")))
    .withColumn("InvoiceQuarter", F.quarter(F.col("InvoiceDate")))

    # silver metadata
    .withColumn("silver_ingested_ts", F.current_timestamp())
)

silver_sales_df.select(
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "UnitPrice",
        "CustomerID",
        "Country",
        "Discount",
        "PaymentMethod",
        "ShippingCost",
        "Category",
        "SalesChannel",
        "ReturnStatus",
        "ShipmentProvider",
        "WarehouseLocation",
        "OrderPriority",
        "_ingestedDate",
        "_ingestedTs",
        "InvoiceCancelledInd",
        "Sales",
        "Profit",
        "InvoiceMonth",
        "InvoiceWeek",
        "InvoiceDay",
        "InvoiceDayofWeek",
        "InvoiceYear",
        "InvoiceQuarter",
        "silver_ingested_ts",
    )

#display(silver_sales_df.limit(10))

In [0]:
#Apply Data Quality Filters
silver_sales_clean_df = (
    silver_sales_df
    .filter(F.col("InvoiceNo").rlike(r"^\d+$"))
    .filter(F.col("InvoiceDate").isNotNull())
    .filter(F.col("CustomerID").isNotNull())
    .filter(F.col("StockCode").isNotNull())
    .filter(F.col("Country").isNotNull())
    .filter(F.col("Discount").isNotNull())
    .filter(F.col("PaymentMethod").isNotNull())
    #.filter(F.col("ShippingCost").isNotNull())
    .filter(F.col("InvoiceCancelledInd").rlike(r"^(true|false)$"))
    .filter(F.col("Quantity") > 0)
    .filter(F.col("UnitPrice") > 0)
    .filter(F.col("Sales") > 0)
    .dropna(how="any", subset=["InvoiceNo", "InvoiceDate", "CustomerID", "StockCode", "Country", "Discount", "PaymentMethod", "ShippingCost", "InvoiceCancelledInd"])
    .dropDuplicates(["InvoiceNo", "InvoiceDate", "CustomerID", "StockCode", "Country", "Discount", "PaymentMethod", "ShippingCost", "InvoiceCancelledInd"])
)

#display(silver_sales_clean_df.limit(10))

In [0]:
#Write Silver Delta Table
silver_sales_clean_df.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable("silver_sales")


In [0]:
# spark.sql("""
# SELECT * FROM silver_sales limit 10
# """).show()

display(spark.table("silver_sales").limit(10))